# PART 03 · 2장 작업형 2 데이터 전처리

교재 211~237쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 01 라이브러리 불러오기

### 1 기본 라이브러리와 import

#### 1. 작업형 2에서 사용하는 기본 라이브러리

#### 2. 기본 import 코드

시험에서 그대로 사용할 기본 import 코드는 다음과 같다.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
print('라이브러리 불러오기 완료')

라이브러리 불러오기 완료


### 2 import 확인과 오류 점검

#### 1. 불러온 이름 확인

불러온 라이브러리가 정상적으로 연결되었는지 이름을 출력해 확인해본다.

In [2]:
print(pd.__name__)
print(train_test_split.__name__)
print(RandomForestClassifier.__name__)
print(roc_auc_score.__name__)

pandas
train_test_split
RandomForestClassifier
roc_auc_score


#### 2. import 오류를 확인하는 기준

## 02 데이터 불러오기

### 1 파일 불러오기와 확인

#### 1. 공통 경로와 세 파일 불러오기

같은 폴더에 있는 파일을 여러 개 읽을 때는 공통 경로를 base 변수에 저장하면 코드가 짧아지고 경로 오타를 줄일 수 있다.

In [3]:
base = 'data/part03_type2/binary_learning_renewal'
train = pd.read_csv(base+'/train.csv')
test = pd.read_csv(base+'/test.csv')
submit = pd.read_csv(base+'/sample_submission.csv')
print(train.shape)
print(test.shape)
print(submit.shape)

(720, 15)
(240, 14)
(240, 1)


#### 2. train 데이터 일부 확인

train에서 주요 컬럼 몇 개를 골라 앞부분을 확인해본다.

In [4]:
cols = ['student_id', 'age_group', 'region', 'course_category',
        'quiz_avg_score', 'renew_next_month']
print(train[cols].head(3).to_string(index=False))

 student_id age_group region course_category  quiz_avg_score  renew_next_month
     100001       30대    강원권             SQL            69.5                 0
     100002       30대    수도권             SQL            72.8                 1
     100003       20대    수도권            머신러닝            65.0                 0


#### 3. test와 제출 예시 파일 확인

test와 sample_submission도 같은 방식으로 확인해본다.

In [5]:
test_cols = ['student_id', 'age_group', 'region',
             'course_category', 'quiz_avg_score']
print(test[test_cols].head(3).to_string(index=False))
print(submit.head().to_string(index=False))

 student_id age_group region course_category  quiz_avg_score
     200001       20대    강원권            클라우드            95.0
     200002       30대    충청권          Python            64.2
     200003       40대    수도권          Python            72.0
 pred
  0.0
  0.0
  0.0
  0.0
  0.0


### 2 파일 관계와 경로 점검

#### 1. 파일 사이의 관계 확인

세 파일 사이의 관계를 컬럼 차집합과 행 수 비교로 확인해본다.

In [6]:
print(set(train.columns) - set(test.columns))
print(set(test.columns)-set(train.columns))
print(len(test) == len(submit))

{'renew_next_month'}
set()
True


#### 2. 경로 오류 점검

## 03 EDA 확인

### 1 확인 항목과 기본 정보

#### 1. 작업형 2에서 먼저 확인할 항목

#### 2. 자료형 확인

모델 입력에는 일반적으로 숫자 형태의 값이 필요하다. object형 컬럼이 있으면 이후 인코딩이 필요하고, 수치형 컬럼은 결측치와 값의 범위를 확인한다.

In [7]:
print(train.dtypes)

student_id                 int64
age_group                 object
region                    object
device_type               object
course_category           object
membership_months          int64
study_minutes_week         int64
completed_lessons          int64
quiz_avg_score           float64
assignment_rate          float64
late_submission_count      int64
community_posts            int64
coupon_used                int64
last_login_days            int64
renew_next_month           int64
dtype: object


#### 3. train과 test의 결측치 확인

train과 test의 결측치를 함께 확인한다. 0보다 큰 컬럼만 골라 출력해본다.

In [8]:
train_missing = train.isnull().sum()
test_missing = test.isnull().sum()
print(train_missing[train_missing > 0])
print(test_missing[test_missing > 0])

device_type       18
quiz_avg_score    39
dtype: int64
device_type        7
quiz_avg_score    14
dtype: int64


### 2 target과 수치형 확인

#### 1. target 분포 확인

target의 클래스 분포를 개수와 비율로 확인해본다.

In [9]:
print(train['renew_next_month'].value_counts().sort_index())
print(train['renew_next_month'].value_counts(normalize=True).sort_index().round(3))

renew_next_month
0    332
1    388
Name: count, dtype: int64
renew_next_month
0    0.461
1    0.539
Name: proportion, dtype: float64


#### 2. target 값의 의미 확인

#### 3. 수치형 컬럼 요약

수치형 컬럼의 분포는 describe( )로 한 번에 요약할 수 있다.

In [10]:
summary_cols = ['membership_months', 'study_minutes_week',
                'quiz_avg_score', 'assignment_rate',
                'last_login_days']
print(train[summary_cols].describe().round(2))

       membership_months  study_minutes_week  quiz_avg_score  assignment_rate  \
count             720.00              720.00          681.00           720.00   
mean               12.38              351.47           71.82             0.68   
std                 6.96              108.17           12.30             0.16   
min                 1.00               44.00           35.50             0.22   
25%                 6.00              275.00           63.40             0.58   
50%                12.00              352.00           71.60             0.69   
75%                19.00              428.25           80.20             0.79   
max                24.00              663.00          100.00             1.00   

       last_login_days  
count           720.00  
mean              7.26  
std               4.69  
min               0.00  
25%               4.00  
50%               6.00  
75%              10.00  
max              28.00  


## 04 target 분리

### 1 X와 y 구분과 확인

#### 1. X와 y의 역할

#### 2. target 존재 여부 확인

분리하기 전에 target 컬럼이 train에 있는지 먼저 확인한다.

In [11]:
target = 'renew_next_month'
print(target in train.columns)
print(target in test.columns)

True
False


### 2 target 분리와 값 처리

#### 1. drop으로 target 분리하기

drop( )으로 target을 분리하고 X와 test의 컬럼이 같아졌는지 확인해본다.

In [12]:
X = train.drop(columns=[target])
y = train[target]
print(X.shape)
print(y.shape)
print(test.shape)
print(set(X.columns)-set(test.columns))
print(set(test.columns)-set(X.columns))

(720, 14)
(720,)
(240, 14)
set()
set()


#### 2. pop으로 target 분리하기

pop( )은 target 컬럼을 꺼내면서 원본 데이터프레임에서 제거한다. 원본 train을 이후에도 확인하려면 먼저 copy( )로 복사본을 만드는 편이 안전하다.

In [13]:
train_copy = train.copy()
y2 = train_copy.pop(target)
X2 = train_copy
print(X2.shape)
print(target in X2.columns)
print(y2.head().to_list())

(720, 14)
False
[0, 1, 0, 1, 0]


pop( )을 사용해도 X2에서 target이 빠지고 y2에 값이 담긴 것을 확인할 수 있다.

> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
target = 'renew_next_month'
X = train.drop(columns=[target])
y = train[target]
```

#### 3. target의 결측값과 비정상 대체값 처리

지도 학습에서 target은 모델이 학습해야 할 정답이다. target이 비어 있으면 해당 행에서 무엇을 맞혀야 하는지 알 수 없으므로 일반적으로 학습에 사용할 수 없다. 입력 변수의 결측치는 중앙값이나 최빈값으로 대체할 수 있지만, target을 같은 방식으로 임의 대체하지 않는다.

문제에서 특정 값을 미기록값이라고 안내했다면 target을 분리하기 전에 유효한 행만 남긴다. 다음 코드는 delivery_minutes가 NaN이거나 0인 행을 제외하라는 조건이 주어진 경우의 예다.

위 코드는 PART 03 1장에서 불러온 회귀 데이터 `reg_train`을 사용한다. 장별 노트북에서는 이어서 실행할 수 있도록 여기서 다시 불러온다.

In [14]:
base_reg = 'data/part03_type2/regression_delivery_minutes'
reg_train = pd.read_csv(base_reg + '/train.csv')
print(reg_train.shape)

(840, 15)


In [15]:
target = 'delivery_minutes'
valid_cond = (
    reg_train[target].notna()
    & (reg_train[target] != 0)
)
train_valid = reg_train.loc[valid_cond].copy()
reg_X = train_valid.drop(columns=[target])
reg_y = train_valid[target]
print('before: ', len(reg_train))
print('after: ', len(train_valid))

before:  840
after:  840


## 05 결측치 처리

### 1 결측치 확인과 대체 기준

#### 1. X와 test의 결측치 확인

X와 test의 결측치를 컬럼별로 확인해본다.

In [16]:
print(X.isnull().sum()[X.isnull().sum() > 0])
print(test.isnull().sum()[test.isnull().sum() > 0])

device_type       18
quiz_avg_score    39
dtype: int64
device_type        7
quiz_avg_score    14
dtype: int64


#### 2. 대체 기준 정하기

정한 기준대로 중앙값과 최빈값을 계산해둔다.

In [17]:
score_median = X['quiz_avg_score'].median()
device_mode = X['device_type'].mode()[0]
print(score_median)
print(device_mode)

71.6
mobile


### 2 결측치 대체와 재확인

#### 1. fillna로 결측치 채우기

train 기준으로 만든 대체값을 X와 test에 같은 방식으로 적용한다.

In [18]:
X['quiz_avg_score'] = X['quiz_avg_score'].fillna(score_median)
test['quiz_avg_score'] = test['quiz_avg_score'].fillna(score_median)
X['device_type'] = X['device_type'].fillna(device_mode)
test['device_type'] = test['device_type'].fillna(device_mode)

#### 2. 처리 결과 재확인

처리한 컬럼과 전체 데이터에 결측치가 남지 않았는지 다시 확인한다.

In [19]:
print(X[['device_type', 'quiz_avg_score']].isnull().sum())
print(test[['device_type', 'quiz_avg_score']].isnull().sum())
print(X.isnull().sum().sum())
print(test.isnull().sum().sum())

device_type       0
quiz_avg_score    0
dtype: int64
device_type       0
quiz_avg_score    0
dtype: int64
0
0


## 06 범주형 변수 인코딩

### 1 범주형 컬럼 확인

#### 1. 범주형 컬럼 찾기

select_dtypes( )로 문자형 컬럼만 골라내 본다.

In [20]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
print(cat_cols)
print(len(cat_cols))

['age_group', 'region', 'device_type', 'course_category']
4


#### 2. 범주 값의 수 확인

### 2 원-핫 인코딩

#### 1. get_dummies로 원-핫 인코딩하기

get_dummies( )로 범주형 컬럼을 0과 1의 컬럼으로 펼쳐 본다.

In [21]:
X_encoded = pd.get_dummies(X, dtype=int)
test_encoded = pd.get_dummies(test, dtype=int)
print(X.shape, X_encoded.shape)
print(test.shape, test_encoded.shape)

(720, 14) (720, 29)
(240, 14) (240, 31)


#### 2. 인코딩 결과 확인

인코딩으로 만들어진 컬럼 이름과 남은 문자형 컬럼을 확인해본다.

In [22]:
dummy_cols = [col for col in X_encoded.columns
              if col.startswith(('age_group_', 'device_type_'))]
print(dummy_cols)
print(X_encoded.select_dtypes(include='object').columns.tolist())

['age_group_10대', 'age_group_20대', 'age_group_30대', 'age_group_40대', 'age_group_50대이상', 'device_type_desktop', 'device_type_mobile', 'device_type_tablet']
[]


## 07 수치형 변수 처리

### 1 수치형 컬럼 확인

#### 1. 수치형 컬럼의 역할 구분

#### 2. 수치형 컬럼과 값의 범위 확인

수치형 컬럼 목록과 값의 범위를 확인해본다.

In [23]:
num_cols = X.select_dtypes(include='number').columns.tolist()
print(num_cols)
print(len(num_cols))
check_cols = ['student_id', 'membership_months', 'study_minutes_week',
              'quiz_avg_score', 'assignment_rate',
              'coupon_used', 'last_login_days']
print(X[check_cols].agg(['min', 'max', 'mean']).T.round(3))

['student_id', 'membership_months', 'study_minutes_week', 'completed_lessons', 'quiz_avg_score', 'assignment_rate', 'late_submission_count', 'community_posts', 'coupon_used', 'last_login_days']
10
                          min       max        mean
student_id          100001.00  100720.0  100360.500
membership_months        1.00      24.0      12.378
study_minutes_week      44.00     663.0     351.467
quiz_avg_score          35.50     100.0      71.805
assignment_rate          0.22       1.0       0.679
coupon_used              0.00       1.0       0.282
last_login_days          0.00      28.0       7.262


### 2 스케일링 적용

#### 1. 스케일링을 고려하는 경우

#### 2. StandardScaler 적용

여기서는 student_id와 coupon_used를 제외한 수치형 컬럼에 StandardScaler를 적용한다. X에서 평균과 표준편차를 계산하고, test에는 X에서 계산한 같은 기준을 적용한다.

In [24]:
from sklearn.preprocessing import StandardScaler
scale_cols = [col for col in num_cols
              if col not in ['student_id', 'coupon_used']]
scaler = StandardScaler()
X_scaled = X.copy()
test_scaled = test.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])
test_scaled[scale_cols] = scaler.transform(test[scale_cols])

#### 3. 스케일링 전후 비교

스케일링 전후의 값을 나란히 출력해 변화를 확인해본다.

In [25]:
view_cols = ['membership_months', 'study_minutes_week',
             'quiz_avg_score', 'assignment_rate',
             'last_login_days']
print(X[view_cols].head(3))
print(X_scaled[view_cols].head(3).round(3))
print(round(X_scaled['study_minutes_week'].mean(), 3))
print(round(X_scaled['study_minutes_week'].std(ddof=0), 3))

   membership_months  study_minutes_week  quiz_avg_score  assignment_rate  \
0                 12                 352            69.5            0.699   
1                 12                 257            72.8            0.756   
2                  2                 158            65.0            0.432   

   last_login_days  
0                9  
1                7  
2                6  
   membership_months  study_minutes_week  quiz_avg_score  assignment_rate  \
0             -0.054               0.005          -0.193            0.130   
1             -0.054              -0.874           0.083            0.498   
2             -1.492              -1.790          -0.569           -1.593   

   last_login_days  
0            0.371  
1           -0.056  
2           -0.269  
0.0
1.0


## 08 train/test 컬럼 불일치 처리

### 1 컬럼 불일치 확인

#### 1. 불일치가 생긴 범주 확인

각 범주형 컬럼에서 test에만 있는 값을 찾아 본다.

In [26]:
for col in cat_cols:
    test_only = set(test[col].unique()) - set(X[col].unique())
    if test_only:
        print(col, test_only)

device_type {'smart_tv'}
course_category {'클라우드'}


#### 2. 인코딩 후 컬럼 차이 확인

인코딩 후 두 데이터의 컬럼 차이를 확인해본다.

In [27]:
print(X_encoded.shape)
print(test_encoded.shape)
print(sorted(set(X_encoded.columns)-set(test_encoded.columns)))
print(sorted(set(test_encoded.columns)-set(X_encoded.columns)))

(720, 29)
(240, 31)
[]
['course_category_클라우드', 'device_type_smart_tv']


### 2 test 컬럼 맞추기

#### 1. X 기준으로 test 컬럼 맞추기

reindex( )를 사용해 test의 컬럼을 X 기준으로 맞춰 본다.

In [28]:
test_encoded = test_encoded.reindex(
    columns=X_encoded.columns,
    fill_value=0
)
print(X_encoded.shape)
print(test_encoded.shape)
print(X_encoded.columns.equals(test_encoded.columns))

(720, 29)
(240, 29)
True


## 09 train_test_split

### 1 검증 데이터와 기본 분할

#### 1. 제공된 test와 검증 데이터의 차이

#### 2. 기본 분할 코드

train_test_split( )으로 학습용과 검증용 데이터를 나눠 본다.

In [29]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)
print(X_train.shape, X_val.shape)
print(y_train.shape, y_val.shape)

(576, 29) (144, 29)
(576,) (144,)


### 2 분할 옵션

#### 1. 주요 인자 확인

#### 2. stratify 적용 결과

stratify=y를 지정하면 분할 후에도 클래스 비율이 유지된다. 세 데이터의 분포를 비교해본다.

In [30]:
print(y.value_counts().sort_index())
print(y_train.value_counts().sort_index())
print(y_val.value_counts().sort_index())

renew_next_month
0    332
1    388
Name: count, dtype: int64
renew_next_month
0    266
1    310
Name: count, dtype: int64
renew_next_month
0    66
1    78
Name: count, dtype: int64


## 10 검증 데이터 구성

### 1 전체 흐름 정리

#### 1. 전체 전처리 흐름

#### 2. 검증 단계와 최종 제출 단계 비교

### 2 검증 데이터 구성

#### 1. 데이터 준비 코드 1

데이터를 불러오고 분할까지 마치는 준비 코드부터 실행한다.

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
base = 'data/part03_type2/binary_learning_renewal'
train = pd.read_csv(base+'/train.csv')
test = pd.read_csv(base+'/test.csv')
target = 'renew_next_month'
X_raw = train.drop(columns=[target, 'student_id']).copy()
y = train[target].copy()
X_test_raw = test.drop(columns=['student_id']).copy()
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

#### 2. 데이터 준비 코드 2

분할된 데이터에 train 기준 결측치 처리를 적용한다.

In [32]:
num_cols = X_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
    X_test_raw[col] = X_test_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
    X_test_raw[col] = X_test_raw[col].fillna(fill_value)

#### 3. 학습용, 검증용 데이터 만들기

분할된 원본 입력에 각각 get_dummies( )를 적용하고 X_train 기준으로 컬럼을 맞춘다.

In [33]:
X_train = pd.get_dummies(X_train_raw)
X_val = pd.get_dummies(X_val_raw)
X_test = pd.get_dummies(X_test_raw)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape)
print(X_train.isnull().sum().sum())
print(X_val.isnull().sum().sum())
print(X_test.isnull().sum().sum())

(576, 28) (144, 28) (240, 28)
(576,) (144,)
0
0
0


### 3 검증 점수와 결과 확인

#### 1. 간단한 모델로 검증 점수 확인

검증 데이터가 어떻게 사용되는지 확인하기 위해 랜덤포레스트 모델을 한 번 학습한다. 모델의 세부 원리와 다른 분류 모델 비교는 다음 장에서 다룬다.

In [34]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
model = RandomForestClassifier(random_state=0)
model.fit(X_train, y_train)
positive_class = 1
positive_index = list(model.classes_).index(positive_class)
pred_prob = model.predict_proba(X_val)[:, positive_index]
pred_class = model.predict(X_val)
print('classes: ', model.classes_)
print('ROC-AUC: ', round(roc_auc_score(y_val, pred_prob), 4))
print('Accuracy: ', round(accuracy_score(y_val, pred_class), 4))

classes:  [0 1]
ROC-AUC:  0.7061
Accuracy:  0.6181


#### 2. 예측 결과 일부 확인

검증 데이터의 실젯값과 예측 클래스, 확률을 나란히 확인해본다.

In [35]:
val_check = pd.DataFrame({
    'actual': y_val.values[:5],
    'pred_class': pred_class[:5],
    'pred_prob': pred_prob[:5].round(2)
})
print(val_check)

   actual  pred_class  pred_prob
0       0           1       0.81
1       0           1       0.51
2       1           1       0.76
3       0           0       0.28
4       0           1       0.52


#### 3. 검증 데이터와 최종 test 구분

> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
```